# Lab Day 19 — GraphRAG với AI Company Corpus
**Sinh viên:** Lương Quốc Dũng — 2A202600601  
**Corpus:** Epoch AI — Data on AI Companies (9 công ty, 6 bảng CSV)  
**Stack:** NetworkX + gpt-5.4-nano (OpenAI) + ChromaDB + sentence-transformers

In [ ]:
import sys, os
sys.path.insert(0, 'src')
os.makedirs('outputs', exist_ok=True)
os.makedirs('.cache', exist_ok=True)
print('Working dir:', os.getcwd())

## 1. Environment Setup

In [ ]:
# Kiểm tra model và kết nối
from config import make_openai_client, OPENAI_MODEL
c = make_openai_client()
r = c.chat.completions.create(model=OPENAI_MODEL,
    messages=[{'role':'user','content':'Say READY'}],
    max_completion_tokens=10)
print(f'Model: {OPENAI_MODEL}')
print(f'Connection: {r.choices[0].message.content}')

## 2. Bước 1 — Indexing: Trích xuất Triples (Entity & Relation Extraction)

In [ ]:
import json
from extract import build_triples

# Dùng cache nếu đã chạy trước, tránh tốn token
payload = build_triples(use_llm=True, use_cache=True)

print(f"Companies : {len(payload['companies'])}")
print(f"Triples   : {payload['counts']}")
print(f"LLM usage : {payload['usage']['total_tokens']} tokens, ~${payload['usage']['est_cost_usd']:.4f}")

In [ ]:
# Xem mẫu triples
triples = payload['triples']
print('=== Structured triples (mẫu) ===')
for t in [x for x in triples if x['kind']=='attr'][:5]:
    print(f"  ({t['subject']}, {t['relation']}, {t['object']})")

print('\n=== Investor triples từ LLM (mẫu) ===')
for t in [x for x in triples if x['relation']=='INVESTED_IN'][:8]:
    print(f"  ({t['subject']}, {t['relation']}, {t['object']})")

## 3. Bước 2 — Construction: Xây dựng Knowledge Graph (NetworkX)

In [ ]:
from graph_build import build_graph, save_graph, draw_graph, print_stats

G = build_graph()
print_stats(G)
save_graph(G)

In [ ]:
# Vẽ và hiển thị Knowledge Graph (Deliverable #2)
from IPython.display import Image
img_path = draw_graph(G)
print(f'Graph saved: {img_path}')
Image(img_path, width=900)

In [ ]:
# Demo 2-hop traversal — tìm common investors giữa OpenAI và Anthropic
import networkx as nx
from graph_build import canonicalize

def common_investors(G, company_a, company_b):
    inv_a = {u for u,v,d in G.in_edges(company_a, data=True) if d['rel']=='INVESTED_IN'}
    inv_b = {u for u,v,d in G.in_edges(company_b, data=True) if d['rel']=='INVESTED_IN'}
    return inv_a & inv_b

common = common_investors(G, 'OpenAI', 'Anthropic')
print('Investors in BOTH OpenAI & Anthropic:', common)

## 4. Bước 3 — Querying: Flat RAG vs GraphRAG

In [ ]:
from flat_rag import FlatRAG
from graph_rag import GraphRAG

flat = FlatRAG()
n_docs = flat.index()
print(f'Flat RAG indexed: {n_docs} documents')

grag = GraphRAG(G=G)
print('GraphRAG ready')

In [ ]:
# Thử nghiệm câu hỏi multi-hop tiêu biểu
test_questions = [
    'Who invested in both OpenAI and Anthropic?',
    'Which AI companies did Amazon invest in?',
    'What product does OpenAI have and how many users does it have?',
]

for q in test_questions:
    r1 = flat.query(q)
    r2 = grag.query(q)
    print(f'\nQ: {q}')
    print(f'Flat RAG : {r1["answer"][:150]}')
    print(f'GraphRAG : {r2["answer"][:150]}')
    print('-'*60)

## 5. Bước 4 — Evaluation: Benchmark 20 câu hỏi

In [ ]:
import pandas as pd
from evaluate import run_benchmark

results = run_benchmark()

In [ ]:
# Hiển thị bảng kết quả
import pandas as pd
df = pd.read_csv('outputs/benchmark_results.csv')

# Summary
flat_acc  = (df['flat_correct']  == 'CORRECT').sum()
graph_acc = (df['graph_correct'] == 'CORRECT').sum()
halluc    = (df['hallucination_caught'] == 'YES').sum()

print(f'Flat RAG  accuracy: {flat_acc}/20 ({flat_acc/20*100:.0f}%)')
print(f'GraphRAG  accuracy: {graph_acc}/20 ({graph_acc/20*100:.0f}%)')
print(f'Hallucination cases caught: {halluc}')
print()

# Bảng đầy đủ
display(df[['id','hop','question','flat_correct','graph_correct','hallucination_caught']].to_string(index=False))

In [ ]:
# Hiển thị cost report
with open('outputs/cost_report.md', encoding='utf-8') as f:
    print(f.read())

## 6. Kết luận

| | Flat RAG | GraphRAG |
|---|---|---|
| Accuracy | 60% | 50% |
| Multi-hop accuracy | 61.5% | 30.8% |
| Hallucination cases caught | — | **2** |
| Token cost | ~$0.004 | ~$0.011 |

**Điểm mạnh GraphRAG:** Câu hỏi cross-entity ("ai đầu tư vào CẢ A lẫn B") — duyệt đồ thị tìm được ngay.  
**Điểm yếu:** Câu hỏi cần sort/aggregate trên graph ("mới nhất", ">1 company") — textualization chưa đủ.  
**Hybrid approach** (GraphRAG + Flat RAG reranking) sẽ tối ưu nhất cho production.